# 1. Prepare and explore Spotify tracks

The root notebook is preserved as an original experiment. This executable copy uses shared helpers in `pipeline_lib.py` so every stage writes to the same isolated run. Launch with `.venv/bin/python run_pipeline.py --profile smoke`.

We clean all raw rows, then optionally take a small stratified sample for testing. A saved train/holdout split is reused throughout the workflow.

In [ ]:
from pipeline_lib import RunContext, prepare_data
ctx = RunContext.from_environment("data_processing")
print("Profile:", ctx.config["profile"])
print("Artifacts:", ctx.run_dir)
import pandas as pd
from IPython.display import Image, display
pd.read_csv(ctx.root / 'spotify_tracks_dataset.csv').head()

Cleaning follows the original notebook: remove missing values and duplicates, convert explicit flags to integers, keep speechiness ≤ 0.7, instrumentalness ≤ 0.9, liveness ≤ 0.8 and nonzero durations. Track identity is retained separately for readable predictions. Original CSV files are never overwritten.

In [ ]:
data_summary = prepare_data(ctx)
display(data_summary)

In [ ]:
train, holdout = ctx.data()
print('Training shape:', train.shape, 'Holdout shape:', holdout.shape)
display(train.head())
display(Image(filename=str(ctx.path('plots', 'data_distribution.png'))))

The holdout stays separate from fitting and parameter selection. Popularity > 70 defines a hit; this is a Spotify label, not a Billboard chart label. Next: classification.